# Phase 3 · Decision Tree classification

This notebook trains and evaluates two Decision Tree classifiers for the Sentinel-2 land-cover dataset: a baseline with the library defaults and a tuned configuration selected in Phase 2.

The train/test split is grouped by polygon, so pixels from the same ground-truth polygon never cross the evaluation boundary. The notebook uses the environment variables provided by the Jupyter Docker service for MinIO, PostgreSQL and MLflow.

In [1]:
import io
import json
import os
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score,
    cohen_kappa_score, confusion_matrix, f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.tree import DecisionTreeClassifier
from sqlalchemy import URL, create_engine, text

DATASET = "T18NWL_20260810T152649"
BANDS = [band.strip() for band in ['B01','B02','B03','B04','B05','B06','B07','B8A','B11','B12']]
SEED = 42
N_SPLITS = 5
OUTPUTS = Path("outputs/phase3")
OUTPUTS.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = {
    0: "No data", 1: "Saturated or defective", 2: "Dark area pixels",
    3: "Cloud shadows", 4: "Vegetation", 5: "Not vegetated",
    6: "Water", 7: "Unclassified", 8: "Cloud medium probability",
    9: "Cloud high probability", 10: "Thin cirrus", 11: "Snow",
}

required = ["DATA_BUCKET", "MLFLOW_S3_ENDPOINT_URL", "TRAIN_DB_HOST",
            "TRAIN_DB_PORT", "TRAIN_DB_NAME", "TRAIN_DB_USER",
            "TRAIN_DB_PASSWORD", "MLFLOW_TRACKING_URI"]
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise RuntimeError(f"Missing Docker environment variables: {missing}")

print("Environment ready")
print("MLflow:", os.environ["MLFLOW_TRACKING_URI"])
print("MinIO endpoint:", os.environ["MLFLOW_S3_ENDPOINT_URL"])
print("Bands:", BANDS)

RuntimeError: Missing Docker environment variables: ['DATA_BUCKET', 'MLFLOW_S3_ENDPOINT_URL', 'TRAIN_DB_HOST', 'TRAIN_DB_PORT', 'TRAIN_DB_NAME', 'TRAIN_DB_USER', 'TRAIN_DB_PASSWORD', 'MLFLOW_TRACKING_URI']

## 1. Load the TSV and recover polygon groups

The TSV is exported by the `extract_training_samples` DAG. Its coordinates are joined to `extracted.polygon_pixels` so the grouped split can be reproduced.

In [ ]:
s3 = boto3.client("s3", endpoint_url=os.environ["MLFLOW_S3_ENDPOINT_URL"])
tsv_key = f"tsv/{DATASET}.tsv"
body = s3.get_object(Bucket=os.environ["DATA_BUCKET"], Key=tsv_key)["Body"].read()
samples = pd.read_csv(io.BytesIO(body), sep="\t", float_precision="round_trip")
expected_columns = ["Latitude", "Longitude", *BANDS, "Class"]
assert list(samples.columns) == expected_columns, samples.columns.tolist()
assert not samples.isna().any().any()

engine = create_engine(URL.create(
    "postgresql+psycopg2",
    username=os.environ["TRAIN_DB_USER"],
    password=os.environ["TRAIN_DB_PASSWORD"],
    host=os.environ["TRAIN_DB_HOST"],
    port=int(os.environ["TRAIN_DB_PORT"]),
    database=os.environ["TRAIN_DB_NAME"],
))
with engine.connect() as connection:
    polygons = pd.read_sql(
        text('SELECT latitude AS "Latitude", longitude AS "Longitude", fid '
             "FROM extracted.polygon_pixels WHERE dataset = :dataset"),
        connection, params={"dataset": DATASET},
    )

samples = samples.merge(polygons, on=["Latitude", "Longitude"],
                        how="left", validate="one_to_one")
assert samples["fid"].notna().all(), "Some samples have no polygon group"
samples["fid"] = samples["fid"].astype(int)
print(f"Loaded {len(samples):,} samples from s3://{os.environ['DATA_BUCKET']}/{tsv_key}")
print(f"Polygon groups: {samples['fid'].nunique():,}")
samples.head()

NameError: name 'boto3' is not defined

## 2. Grouped train/test split

One of five stratified group folds is held out. The class labels used for macro metrics are restricted to classes present in the test set.

In [ ]:
X = samples[BANDS]
y = samples["Class"]
groups = samples["fid"]
splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
train_index, test_index = next(splitter.split(X, y, groups))

X_train = X.iloc[train_index].reset_index(drop=True)
y_train = y.iloc[train_index].reset_index(drop=True)
groups_train = groups.iloc[train_index].reset_index(drop=True)
X_test = X.iloc[test_index].reset_index(drop=True)
y_test = y.iloc[test_index].reset_index(drop=True)
groups_test = groups.iloc[test_index].reset_index(drop=True)

assert not set(groups_train) & set(groups_test), "A polygon appears in train and test"
EVAL_LABELS = sorted(y_test.unique())
ALL_LABELS = sorted(y.unique())
split_summary = pd.DataFrame({
    "train_samples": y_train.value_counts(),
    "test_samples": y_test.value_counts(),
    "train_polygons": groups_train.groupby(y_train).nunique(),
    "test_polygons": groups_test.groupby(y_test).nunique(),
}).fillna(0).astype(int).sort_index()
split_summary.to_csv(OUTPUTS / "split.csv")
print(f"Train: {len(y_train):,} · test: {len(y_test):,}")
print("Evaluated classes:", EVAL_LABELS)
split_summary

## 3. Train baseline and tuned Decision Trees

The tuned configuration comes from Phase 2: `min_samples_leaf=20`, `criterion='gini'`, no class weighting and unlimited depth. A minimum leaf size controls overfitting while preserving the tree's interpretability.

In [ ]:
MODELS = {
    "baseline": ("Decision Tree · baseline", DecisionTreeClassifier(random_state=SEED)),
    "tuned": ("Decision Tree · tuned", DecisionTreeClassifier(
        criterion="gini", max_depth=None, min_samples_leaf=20,
        class_weight=None, random_state=SEED,
    )),
}

def evaluate(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=EVAL_LABELS, average=None, zero_division=0
    )
    macro = precision_recall_fscore_support(
        y_true, y_pred, labels=EVAL_LABELS, average="macro", zero_division=0
    )
    return {
        "overall_accuracy": float(accuracy_score(y_true, y_pred)),
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision_macro": float(macro[0]),
        "recall_macro": float(macro[1]),
        "f1_macro": float(macro[2]),
    }, pd.DataFrame({"precision": precision, "recall": recall, "f1": f1}, index=EVAL_LABELS)

mlflow.set_experiment(os.environ["MLFLOW_EXPERIMENT_NAME"])
SKOPS_TRUSTED_TYPES = ["sklearn.tree._tree.Tree"]
results = []
models = {}
for key, (name, model) in MODELS.items():
    model.fit(X_train, y_train)
    predicted = model.predict(X_test)
    metrics, per_class = evaluate(y_test, predicted)
    matrix = pd.DataFrame(
        confusion_matrix(y_test, predicted, labels=ALL_LABELS),
        index=ALL_LABELS, columns=ALL_LABELS,
    )
    matrix.index.name, matrix.columns.name = "reference", "predicted"
    matrix.to_csv(OUTPUTS / f"confusion_matrix_{key}.csv")
    per_class.insert(0, "name", per_class.index.map(CLASS_NAMES))
    per_class.index.name = "Class"
    per_class.to_csv(OUTPUTS / f"per_class_{key}.csv")
    figure, axis = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay.from_predictions(
        y_test, predicted, labels=EVAL_LABELS, normalize="true",
        values_format=".2f", cmap="Blues", colorbar=False, ax=axis,
    )
    axis.set(title=f"{name} · OA {metrics['overall_accuracy']:.3f} · F1 {metrics['f1_macro']:.3f}",
             xlabel="Predicted class", ylabel="Reference class")
    figure.tight_layout()
    figure.savefig(OUTPUTS / f"confusion_matrix_{key}.png", dpi=150)
    plt.show()
    models[key] = model
    results.append({"key": key, "model": name, **metrics,
                    "train_accuracy": float(accuracy_score(y_train, model.predict(X_train)))})

    with mlflow.start_run(run_name=f"phase3-{key}") as run:
        mlflow.log_params({
            "phase": 3, "model": name, "dataset": DATASET,
            "bands": ",".join(BANDS), "seed": SEED,
            "split": f"StratifiedGroupKFold by polygon, 1 of {N_SPLITS} folds held out",
            **{f"tree_{param}": value for param, value in model.get_params().items()
               if param in {"criterion", "max_depth", "min_samples_leaf", "class_weight"}},
        })
        mlflow.log_metrics(metrics)
        mlflow.log_metric("train_accuracy", results[-1]["train_accuracy"])
        for artifact in (f"confusion_matrix_{key}.png", f"confusion_matrix_{key}.csv",
                         f"per_class_{key}.csv"):
            mlflow.log_artifact(OUTPUTS / artifact)
        mlflow.sklearn.log_model(model, name="model",
                                 skops_trusted_types=SKOPS_TRUSTED_TYPES)
        results[-1]["mlflow_run_id"] = run.info.run_id

summary = pd.DataFrame(results).sort_values("f1_macro", ascending=False).reset_index(drop=True)
summary.to_csv(OUTPUTS / "summary.csv", index=False)
(OUTPUTS / "metrics.json").write_text(json.dumps({
    "dataset": DATASET, "train_samples": len(y_train), "test_samples": len(y_test),
    "evaluated_classes": EVAL_LABELS, "models": results,
}, indent=2))
summary.drop(columns=["key", "mlflow_run_id"]).round(4)

## 4. Interpretation

The final table and figures are saved in `outputs/phase3/`. Compare the tuned tree with the baseline using macro-F1 and balanced accuracy, not only overall accuracy, because the land-cover classes are imbalanced.